# BeyondBench: Basic Evaluation Walkthrough

This notebook walks through a basic BeyondBench evaluation from scratch.

**What you'll learn:**
- How to run evaluations via the CLI
- How to use the Python API directly
- How to read and interpret results

**Prerequisites:**
```bash
pip install beyondbench[openai]   # for the API example
# or: pip install beyondbench[vllm]  # for the local model example
```

## 1. Verify Installation

In [ ]:
# Check BeyondBench is installed
!beyondbench --version

In [ ]:
# List available tasks
!beyondbench list-tasks --suite easy

## 2. CLI Evaluation: OpenAI API Model

The simplest way to run an evaluation is via the CLI. Here we evaluate GPT-4o on a small subset of easy tasks.

In [ ]:
import os

# Set your API key (or set OPENAI_API_KEY in your environment)
# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"

# Quick 5-sample evaluation on sum and mean tasks
!beyondbench evaluate \
    --model-id gpt-4o \
    --api-provider openai \
    --tasks sum --tasks mean \
    --datapoints 5 \
    --output-dir /tmp/bb_basic_demo \
    --log-level WARNING

## 3. CLI Evaluation: Local Model (vLLM)

For local GPU inference, use the `--backend vllm` option.

In [ ]:
# Evaluate a small local model (requires vllm and a CUDA GPU)
# Uncomment and modify the cuda-device as needed

# !beyondbench evaluate \
#     --model-id Qwen/Qwen2.5-1.5B-Instruct \
#     --backend vllm \
#     --cuda-device cuda:0 \
#     --gpu-memory-utilization 0.45 \
#     --suite easy \
#     --datapoints 5 \
#     --output-dir /tmp/bb_local_demo

print("Uncomment the cell above to run local evaluation")

## 4. Using a Config File

Config files are the preferred way to run reproducible evaluations.

In [ ]:
# View a bundled config file
!cat /data/wang/gks/Development/BeyondBench/beyondbench/configs/quick_test.yaml

In [ ]:
# Run using a config file
# !beyondbench run-config /data/wang/gks/Development/BeyondBench/beyondbench/configs/quick_test.yaml

print("Uncomment above to run with config file")

## 5. Python API: Programmatic Evaluation

You can also run evaluations entirely from Python without using the CLI.

In [ ]:
from beyondbench.models.model_handler import ModelHandler
from beyondbench.core.evaluation_engine import EvaluationEngine

# Initialize the model handler
# For OpenAI:
import os
api_key = os.environ.get("OPENAI_API_KEY", "")

if not api_key:
    print("Set OPENAI_API_KEY to run this cell")
else:
    handler = ModelHandler(
        model_id="gpt-4o",
        api_provider="openai",
        api_key=api_key,
    )
    print("Model info:", handler.get_model_info())

In [ ]:
# Run evaluation programmatically
if api_key:
    engine = EvaluationEngine(
        model_handler=handler,
        output_dir="/tmp/bb_api_demo",
        store_details=False,
    )

    results = engine.run_evaluation(
        suite="easy",
        tasks=["sum", "mean", "sorting"],
        datapoints=10,
        folds=1,
        temperature=0.0,
        seed=42,
    )

    print("Evaluation complete!")
    print(f"Tasks run: {list(results['task_results'].keys())}")
else:
    print("Skipped: no API key")

## 6. Reading and Interpreting Results

Results are saved as JSON files. Let's load and inspect them.

In [ ]:
import json
import os

# Load results from the CLI run (or API run)
results_dir = "/tmp/bb_basic_demo"
results_file = os.path.join(results_dir, "final_results.json")

if os.path.exists(results_file):
    with open(results_file) as f:
        results = json.load(f)
    print("Loaded results successfully")
    print("Top-level keys:", list(results.keys()))
else:
    print(f"No results file found at {results_file}")
    print("Run the evaluation cell first.")

In [ ]:
# Display per-task accuracy
if os.path.exists(results_file):
    print("Per-task accuracy:")
    print("-" * 40)
    for task_name, task_data in results.get("task_results", {}).items():
        if isinstance(task_data, list) and task_data:
            acc = sum(m.get("accuracy", 0) for m in task_data) / len(task_data)
        elif isinstance(task_data, dict):
            summary = task_data.get("summary", {})
            acc = summary.get("avg_accuracy", task_data.get("overall_accuracy", 0))
        else:
            acc = 0.0
        print(f"  {task_name:30s}: {acc:.1%}")

In [ ]:
# Show model info
if os.path.exists(results_file):
    model_info = results.get("model_info", {})
    print("Model info:")
    for k, v in model_info.items():
        print(f"  {k}: {v}")

    summary = results.get("summary", {})
    if summary:
        print(f"\nOverall average accuracy: {summary.get('avg_accuracy', 0):.1%}")
        print(f"Total duration: {summary.get('total_duration', 0):.1f}s")

## 7. Output File Structure

BeyondBench writes the following files:

In [ ]:
# Explore the output directory
import os

for root, dirs, files in os.walk("/tmp/bb_basic_demo"):
    # Skip __pycache__ and hidden dirs
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace("/tmp/bb_basic_demo", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for file in files:
        fpath = os.path.join(root, file)
        size = os.path.getsize(fpath)
        print(f"{sub_indent}{file}  ({size} bytes)")

## Summary

You've seen how to:
- Run a BeyondBench evaluation via CLI: `beyondbench evaluate --model-id ... --suite easy`
- Use a YAML config file: `beyondbench run-config config.yaml`
- Call the Python API: `ModelHandler` + `EvaluationEngine`
- Load and interpret `final_results.json`

**Next notebooks:**
- `02_custom_tasks.ipynb` — Create your own evaluation task
- `03_model_comparison.ipynb` — Compare multiple models side by side
- `04_result_analysis.ipynb` — Deep-dive analysis and visualization